# Question accuracy for one model

Which of several questions (prompts) gets the most images right, compared with the annotated ground truth?

**How to use:** run the VQA with ammico on one of the course datasets and save the result as a csv file. Fill in the two paths in the next cell and run all cells. Tables and plots are saved in `comparison_results/` next to the (first) result file.

In [ ]:
# ============ SETTINGS: the only cell you need to change ============
RESULTS_FILE = "../my_results/emotions_vqa.csv"  # csv file with the answers of ONE model (your ammico output)
IMAGE_DIR = "../dataset_emotions"  # folder with the images

**Datasets** – the dataset (emotions, politicians or symbols) is detected from the image names in the result file, so only the paths change. The paths are relative to this notebook; in the workshop folder the images are in `dataset_emotions/`, `dataset_politicians/` and `dataset_symbols/`, next to `ground_truth/`:
```python
RESULTS_FILE = "../my_results/politicians_vqa.csv"
IMAGE_DIR = "../dataset_politicians"

RESULTS_FILE = "../my_results/symbols_vqa.csv"
IMAGE_DIR = "../dataset_symbols"
```
The result file can be
- an ammico output: `filename` + `vqa` (a list with one answer per question, in the order of the question file), optionally `model`; without a `model` column, the file name is used as the model name,
- a table with `filename` + one column per model (each cell a list with the answers),
- the coded table of the symbols package (`<model>__q<i>_raw` / `<model>__q<i>_code`).

The images can be in one flat folder; they are only needed for the image examples. Questions, ground truth and expected answers are defined in `DATASETS` in `vqa_eval_utils.py`; the ground-truth and question files are read from `ground_truth/` next to the image folder (or from the `*_output/` folders of the course repository). The answers must be to the questions in these files, in the same order. A file with several models (e.g. the coded symbols table) only works in `question_accuracy_models.ipynb`.

## 1. Data and questions

In [ ]:
import vqa_eval_utils as U

run = U.load(RESULTS_FILE, IMAGE_DIR, "question_accuracy_single_model")
assert len(run.models) == 1, f"the file contains several models {run.models}: use question_accuracy_models.ipynb"
U.show_questions(run)

## 2. From free text to answers
Each answer is mapped to a category (yes / no / dk, …) by the rules of its question type. `exact`, `first` and `package_code` are safe; the keyword rules (`kw_*`) are heuristics, so read the sample of raw answers below against their `normalised` value.

In [ ]:
U.show_normalisation(run)

## 3. Accuracy per question
**Accuracy** = share of images whose (normalised) answer matches the expected answer from the ground truth. DK, unclear and unparsed answers count as **wrong**; images without an expected answer (e.g. label `unclear`) are not scored.
The **majority baseline** is the accuracy of always giving the most frequent expected answer – a question is only useful if it beats it clearly. Error bars are 95% Wilson confidence intervals.

In [ ]:
acc = U.show_accuracy(run)

## 4. Accuracy per image group
The same accuracy for each image group: a low value shows which kind of image makes a question fail.

In [ ]:
U.show_groups(run, acc)

## 5. Hardest images
The images with the largest share of wrong answers. For each image: the image, the answer generated for every question (`read as` = the normalised answer that is scored), the expected answer, and which answers are wrong. Errors on every question point to a hard image or a questionable label; errors on some questions point to the wording.

In [ ]:
U.show_hardest_images(run, n=6)  # n = number of images shown